# CGC + Chiller Test

This test is for testing the water supply from the chiller to all water cooled cgc devices.
Devices used: PSU 1-4, SwitchA and SwitchB, Lauda CGC Chiller
Test: Activate Chiller and monitor CGC devices temperature over time


## Import, Setup Logger, Create Instances

In [ ]:
# Import required modules
import sys
import os
import logging
from datetime import datetime
from pathlib import Path

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

from devices.cgc.psu.psu import PSU
from devices.cgc.sw.sw import SW
from devices.cgc.sw_HR.sw_HR import SWHR
from devices.chiller.chiller import Chiller, ChillerCommands

In [ ]:
# Setup external logger
repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"016_Temp_Check_GC_Chiller_{timestamp}.log"

logger = logging.getLogger(f"016_Temp_Check_GC_Chiller_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(console_handler)

logger.info("Logger initialized.")
print(f"Log file: {log_file}")

## Create Device Instances

In [ ]:
psu1 = PSU("psu1", com=3, port=0, logger=logger)

In [ ]:
psu2 = PSU("psu2", com=4, port=1, logger=logger)

In [ ]:
psu3 = PSU("psu3", com=5, port=2, logger=logger)

In [ ]:
psu4 = PSU("psu4", com=6, port=3, logger=logger)

In [ ]:
swA = SWHR("swA", com=7, stream=0, logger=logger)

In [ ]:
swB = SW("swB", com=8, port=0, logger=logger)

In [ ]:
chiller = Chiller("cgc_chiller", port="COM23", baudrate=115200, logger=logger)

## Connect All Devices

In [ ]:
psu1.connect()
psu1.set_comspeed(230400)

In [ ]:
psu2.connect()
psu2.set_comspeed(230400)

In [ ]:
psu3.connect()
psu3.set_comspeed(230400)

In [ ]:
psu4.connect()
psu4.set_comspeed(230400)

In [ ]:
swA.connect()
swA.set_comspeed(230400)

In [ ]:
swB.connect()
swB.set_comspeed(230400)

In [ ]:
chiller.connect()

## Configure and Start Chiller

In [ ]:
chiller.read_pump_level()

In [ ]:
chiller.set_temperature(8.0)

In [ ]:
chiller.start_device()

## Manual Testing of Temperature function Calls

In [ ]:
psu1.get_sensor_data()

In [ ]:
psu2.get_sensor_data()

In [ ]:
psu3.get_sensor_data()

In [ ]:
psu4.get_sensor_data()

In [ ]:
swA.get_sensor_data()

In [ ]:
swB.get_sensor_data()

In [ ]:
chiller.read_temp()

## Temperature Monitoring Loop

In [ ]:
import time
import numpy as np

interval = 1  # seconds

timestamps = []
data = {
    "psu1": [], "psu2": [], "psu3": [], "psu4": [],
    "swA": [], "swB": [],
    "chiller": [],
}

try:
    while True:
        t = time.time()
        timestamps.append(t)

        for name, psu in [("psu1", psu1), ("psu2", psu2), ("psu3", psu3), ("psu4", psu4)]:
            status, t0, t1, t2 = psu.get_sensor_data()
            if status == psu.NO_ERR:
                data[name].append([t0, t1, t2])
                logger.info(f"{name} Sensor0={t0:.1f} Sensor1={t1:.1f} Sensor2={t2:.1f} degC")
            else:
                data[name].append([np.nan, np.nan, np.nan])

        for name, sw in [("swA", swA), ("swB", swB)]:
            status, t0, t1, t2 = sw.get_sensor_data()
            if status == sw.NO_ERR:
                data[name].append([t0, t1, t2])
                logger.info(f"{name} Sensor0={t0:.1f} Sensor1={t1:.1f} Sensor2={t2:.1f} degC")
            else:
                data[name].append([np.nan, np.nan, np.nan])

        temp = chiller.read_temp()
        if temp is not None:
            data["chiller"].append(temp)
            logger.info(f"chiller Temp={temp:.2f} degC")
        else:
            data["chiller"].append(np.nan)

        time.sleep(interval)
except KeyboardInterrupt:
    logger.info("Temperature monitoring stopped")
    print(f"Collected {len(timestamps)} samples")

In [ ]:
save_file = log_dir / f"016_Temp_Data_{timestamp}.npz"

np.savez(
    save_file,
    timestamps=np.array(timestamps),
    psu1=np.array(data["psu1"]),
    psu2=np.array(data["psu2"]),
    psu3=np.array(data["psu3"]),
    psu4=np.array(data["psu4"]),
    swA=np.array(data["swA"]),
    swB=np.array(data["swB"]),
    chiller=np.array(data["chiller"]),
)
print(f"Saved to {save_file}")

In [ ]:
import matplotlib.pyplot as plt

t = (np.array(timestamps) - timestamps[0]) / 60  # minutes
arr_chiller = np.array(data["chiller"])

fig, axes = plt.subplots(6, 1, sharex=True, figsize=(12, 14))

# PSU 1-4
for ax, name in zip(axes[:4], ["psu1", "psu2", "psu3", "psu4"]):
    arr = np.array(data[name])
    ax.plot(t, arr[:, 0], label="Sensor 0")
    ax.plot(t, arr[:, 1], label="Sensor 1")
    ax.plot(t, arr[:, 2], label="Sensor 2")
    ax.plot(t, arr_chiller, label="Chiller", alpha=0.3, color="gray", linestyle="--")
    ax.set_ylabel("Temp (degC)")
    ax.set_title(name)
    ax.legend(loc="upper right")
    ax.grid(True)

# swA - first two sensors
arr_swA = np.array(data["swA"])
axes[4].plot(t, arr_swA[:, 0], label="Sensor 0")
axes[4].plot(t, arr_swA[:, 1], label="Sensor 1")
axes[4].plot(t, arr_chiller, label="Chiller", alpha=0.3, color="gray", linestyle="--")
axes[4].set_ylabel("Temp (degC)")
axes[4].set_title("swA")
axes[4].legend(loc="upper right")
axes[4].grid(True)

# swB - last two sensors
arr_swB = np.array(data["swB"])
axes[5].plot(t, arr_swB[:, 1], label="Sensor 1")
axes[5].plot(t, arr_swB[:, 2], label="Sensor 2")
axes[5].plot(t, arr_chiller, label="Chiller", alpha=0.3, color="gray", linestyle="--")
axes[5].set_ylabel("Temp (degC)")
axes[5].set_title("swB")
axes[5].legend(loc="upper right")
axes[5].grid(True)

axes[5].set_xlabel("Time (min)")
fig.tight_layout()
plt.savefig(log_dir / f"016_Temp_Plot_{timestamp}.png", dpi=150)
plt.show()

## Stop Chiller and Disconnect

In [ ]:
chiller.stop_device()

In [ ]:
chiller.set_temperature(18.0)

In [ ]:
psu1.disconnect()
psu2.disconnect()
psu3.disconnect()
psu4.disconnect()
swA.disconnect()
swB.disconnect()

In [ ]:
chiller.disconnect()